# Tree-based Model Experiments for Healthcare Risk Prediction

This notebook documents and executes the model experiments for the project **A Comparative Study of Tree-based Models for Healthcare Risk Prediction: Cross-validation and SHAP-based Stability Analysis**.

The notebook focuses on the AI Engineer (Model) scope:

- **RQ1:** How do AdaBoost, XGBoost, and LightGBM compare in healthcare risk prediction across multiple datasets?
- **RQ2:** How stable is the predictive performance of these models across cross-validation folds?

**RQ3 (SHAP-based feature-ranking stability) is analyzed separately.**

Reusable implementation is kept in `src/`, while this notebook provides the experiment setup, execution, results, and interpretation.

## 1. Environment and imports

The experiment configuration is stored in `config.yaml`. Model creation, evaluation metrics, and the reusable experiment runner are imported from `src/`.

In [6]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

# Make the notebook work whether it is launched from the repository root
# or from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "config.yaml"
RESULT_DIR = PROJECT_ROOT / "results" / "model_evaluation"

from src.experiment_config import load_experiment_config
from src.experiments.run_models import run_experiments

print(f"Project root: {PROJECT_ROOT}")
print(f"Config:       {CONFIG_PATH}")
print(f"Results:      {RESULT_DIR}")

Project root: D:\Cert AIO\Mod3\healthcare-risk-prediction
Config:       D:\Cert AIO\Mod3\healthcare-risk-prediction\config.yaml
Results:      D:\Cert AIO\Mod3\healthcare-risk-prediction\results\model_evaluation


## 2. Experiment setup

The same fixed configuration is used for all three datasets so that differences in performance reflect model and dataset behavior rather than separate per-dataset hyperparameter searches. The predefined stratified 5-fold assignments generated by the data preprocessing pipeline are used for all three datasets. The fold labels are loaded from data/processed/kfold_indices.csv and are reused consistently across all models. PR-AUC is the primary ranking metric because some datasets are strongly class-imbalanced.

In [7]:
config, _ = load_experiment_config(CONFIG_PATH)
experiment = config["experiment"]

setup_table = pd.DataFrame([
    {"Setting": "Random seed", "Value": experiment["random_state"]},
    {"Setting": "CV strategy", "Value": f"Stratified {experiment['cv']['n_splits']}-fold CV"},
    {"Setting": "Shuffle", "Value": experiment["cv"]["shuffle"]},
    {"Setting": "Decision threshold", "Value": experiment["decision_threshold"]},
    {"Setting": "Primary metric", "Value": experiment["primary_metric"]},
    {"Setting": "Balanced sample weights", "Value": experiment["balance_training"]},
])

display(setup_table)

,Setting,Value
0,Random seed,42
1,CV strategy,Stratified 5-fold CV
2,Shuffle,True
3,Decision threshold,0.5
4,Primary metric,pr_auc
5,Balanced sample weights,True


### 2.1 Datasets

In [8]:
dataset_table = pd.DataFrame([
    {
        "Dataset key": key,
        "Dataset": value["name"],
        "Development file": value["train_path"],
        "Test file": value["test_path"],
        "Target": value["target_column"],
    }
    for key, value in config["datasets"].items()
])

display(dataset_table)

,Dataset key,Dataset,Development file,Test file,Target
0,dataset1,Personal Key Indicators of Heart Disease (2022),data/processed/dataset1/train.csv,data/processed/dataset1/test.csv,HadHeartAttack
1,dataset2,Heart Failure Prediction,data/processed/dataset2/train.csv,data/processed/dataset2/test.csv,HeartDisease
2,dataset3,Heart Disease Health Indicators (BRFSS 2015),data/processed/dataset3/train.csv,data/processed/dataset3/test.csv,HeartDiseaseorAttack


### 2.2 Models and hyperparameters

In [9]:
hyperparameter_rows = []
for model_key, model_cfg in config["models"].items():
    for parameter, value in model_cfg["params"].items():
        if isinstance(value, dict):
            value = ", ".join(f"{k}={v}" for k, v in value.items())
        hyperparameter_rows.append({
            "Model": model_cfg.get("display_name", model_key),
            "Parameter": parameter,
            "Value": value,
        })

hyperparameters = pd.DataFrame(hyperparameter_rows)
display(hyperparameters)

,Model,Parameter,Value
0,AdaBoost,estimator,max_depth=1
1,AdaBoost,n_estimators,200
2,AdaBoost,learning_rate,0.1
3,XGBoost,objective,binary:logistic
4,XGBoost,eval_metric,logloss
5,XGBoost,n_estimators,300
6,XGBoost,learning_rate,0.05
7,XGBoost,max_depth,4
8,XGBoost,min_child_weight,1
9,XGBoost,subsample,0.8


## 3. Execute the experiments

The cell below calls the reusable experiment runner in `src/experiments/run_models`. It performs the same workflow for every dataset/model pair:

1. Load the processed development and untouched test splits.
2. Load the predefined 5-fold assignments from kfold_indices.csv and use each fold once as the validation partition.
3. Train a fresh model in every fold.
4. Compute ROC-AUC, PR-AUC, Recall, and F1 on each validation fold.
5. Refit the model on the full development split and evaluate once on the hold-out test set.
6. Save fold-level, summary, comparison, and metadata files to `results/model_evaluation/`.

`RUN_EXPERIMENTS=True` reproduces the full experiment. After a successful run, it can be changed to `False` when reopening the notebook simply to inspect previously saved results.

In [10]:
RUN_EXPERIMENTS = True

if RUN_EXPERIMENTS:
    output_paths = run_experiments(config_path=CONFIG_PATH)
    print("\nExperiment completed successfully.")
    for name, path in output_paths.items():
        print(f"{name:28s}: {path}")
else:
    print("Skipping execution and using existing files in:", RESULT_DIR)


=== Personal Key Indicators of Heart Disease (2022) (dataset1) ===
Development: (353653, 131) | Test: (88414, 131) | positive rate=0.0568/0.0568
  -> AdaBoost
    Fold 1/5 | roc_auc=0.8774, pr_auc=0.3969, recall=0.6853, f1=0.3890 | fit=123.54s
    Fold 2/5 | roc_auc=0.8684, pr_auc=0.3760, recall=0.6794, f1=0.3677 | fit=103.18s
    Fold 3/5 | roc_auc=0.8738, pr_auc=0.3910, recall=0.6914, f1=0.3725 | fit=68.20s
    Fold 4/5 | roc_auc=0.8669, pr_auc=0.3783, recall=0.6629, f1=0.3798 | fit=72.87s
    Fold 5/5 | roc_auc=0.8713, pr_auc=0.3845, recall=0.6781, f1=0.3733 | fit=65.45s
    Hold-out test | roc_auc=0.8695, pr_auc=0.3860, recall=0.6705, f1=0.3657 | fit=74.63s
  -> XGBoost
    Fold 1/5 | roc_auc=0.8922, pr_auc=0.4245, recall=0.7882, f1=0.3358 | fit=4.94s
    Fold 2/5 | roc_auc=0.8850, pr_auc=0.4081, recall=0.7737, f1=0.3337 | fit=2.31s
    Fold 3/5 | roc_auc=0.8892, pr_auc=0.4203, recall=0.7880, f1=0.3320 | fit=2.39s
    Fold 4/5 | roc_auc=0.8852, pr_auc=0.4059, recall=0.7722, f1=0.3

## 4. Load experiment outputs

In [11]:
fold_metrics = pd.read_csv(RESULT_DIR / "fold_metrics.csv")
cv_summary = pd.read_csv(RESULT_DIR / "cv_summary.csv")
test_metrics = pd.read_csv(RESULT_DIR / "test_metrics.csv")
model_comparison = pd.read_csv(RESULT_DIR / "model_comparison.csv")
overall_comparison = pd.read_csv(RESULT_DIR / "overall_model_comparison.csv")

print("Fold-level rows:", len(fold_metrics))
print("CV summary rows: ", len(cv_summary))
print("Test rows:       ", len(test_metrics))

Fold-level rows: 45
CV summary rows:  9
Test rows:        9


### 4.1 Dataset characteristics observed by the model pipeline

In [12]:
dataset_summary = (
    test_metrics[[
        "dataset_key", "dataset_name", "n_development", "n_test",
        "development_positive_rate", "test_positive_rate"
    ]]
    .drop_duplicates()
    .sort_values("dataset_key")
    .reset_index(drop=True)
)

dataset_summary["development_positive_rate"] = (100 * dataset_summary["development_positive_rate"]).round(2)
dataset_summary["test_positive_rate"] = (100 * dataset_summary["test_positive_rate"]).round(2)
dataset_summary = dataset_summary.rename(columns={
    "dataset_key": "Dataset key",
    "dataset_name": "Dataset",
    "n_development": "Development samples",
    "n_test": "Test samples",
    "development_positive_rate": "Development positive (%)",
    "test_positive_rate": "Test positive (%)",
})

display(dataset_summary)

,Dataset key,Dataset,Development samples,Test samples,Development positive (%),Test positive (%)
0,dataset1,Personal Key Indicators of Heart Disease (2022),353653,88414,5.68,5.68
1,dataset2,Heart Failure Prediction,734,184,55.31,55.43
2,dataset3,Heart Disease Health Indicators (BRFSS 2015),183824,45957,10.32,10.32


## 5. RQ1 - Comparison of predictive performance

For RQ1, the models are compared using the mean 5-fold CV metrics within each dataset. PR-AUC is used as the primary ranking metric, with ROC-AUC, Recall, and F1 reported as complementary measures.

In [13]:
rq1_columns = [
    "dataset_name", "model_name",
    "roc_auc_mean", "pr_auc_mean", "recall_mean", "f1_mean",
    "cv_rank"
]

rq1_table = model_comparison[rq1_columns].copy()
rq1_table = rq1_table.sort_values(["dataset_name", "cv_rank"])
for col in ["roc_auc_mean", "pr_auc_mean", "recall_mean", "f1_mean"]:
    rq1_table[col] = rq1_table[col].round(4)

display(rq1_table)

,dataset_name,model_name,roc_auc_mean,pr_auc_mean,recall_mean,f1_mean,cv_rank
6,Heart Disease Health Indicators (BRFSS 2015),LightGBM,0.8378,0.3797,0.7955,0.3822,1
7,Heart Disease Health Indicators (BRFSS 2015),XGBoost,0.8386,0.3797,0.8010,0.3812,2
8,Heart Disease Health Indicators (BRFSS 2015),AdaBoost,0.8324,0.3678,0.7727,0.3826,3
3,Heart Failure Prediction,XGBoost,0.9263,0.9244,0.8745,0.8755,1
4,Heart Failure Prediction,AdaBoost,0.9242,0.9227,0.8449,0.8578,2
5,Heart Failure Prediction,LightGBM,0.9259,0.9161,0.8843,0.8801,3
0,Personal Key Indicators of Heart Disease (2022),XGBoost,0.8873,0.4136,0.7798,0.3324,1
1,Personal Key Indicators of Heart Disease (2022),LightGBM,0.8871,0.4133,0.7770,0.3351,2
2,Personal Key Indicators of Heart Disease (2022),AdaBoost,0.8715,0.3853,0.6794,0.3765,3


### 5.1 Cross-dataset comparison

In [14]:
overall_cols = [
    "model_name", "mean_dataset_rank",
    "pr_auc_mean", "roc_auc_mean", "recall_mean", "f1_mean"
]

overall_table = overall_comparison[overall_cols].copy()
for col in overall_cols[1:]:
    overall_table[col] = overall_table[col].round(4)

display(overall_table)

,model_name,mean_dataset_rank,pr_auc_mean,roc_auc_mean,recall_mean,f1_mean
0,XGBoost,1.3333,0.5725,0.8841,0.8185,0.5297
1,LightGBM,2.0000,0.5697,0.8836,0.8189,0.5324
2,AdaBoost,2.6667,0.5586,0.8761,0.7657,0.5389


In [15]:
best_rank_row = overall_comparison.sort_values("mean_dataset_rank").iloc[0]
best_pr_row = overall_comparison.sort_values("pr_auc_mean", ascending=False).iloc[0]
best_roc_row = overall_comparison.sort_values("roc_auc_mean", ascending=False).iloc[0]
best_recall_row = overall_comparison.sort_values("recall_mean", ascending=False).iloc[0]
best_f1_row = overall_comparison.sort_values("f1_mean", ascending=False).iloc[0]

rq1_analysis = f"""
### RQ1 Analysis

- **{best_rank_row['model_name']}** achieved the best overall mean dataset rank ({best_rank_row['mean_dataset_rank']:.2f}).
- The highest mean **PR-AUC** was achieved by **{best_pr_row['model_name']}** ({best_pr_row['pr_auc_mean']:.4f}).
- The highest mean **ROC-AUC** was achieved by **{best_roc_row['model_name']}** ({best_roc_row['roc_auc_mean']:.4f}).
- The highest mean **Recall** was achieved by **{best_recall_row['model_name']}** ({best_recall_row['recall_mean']:.4f}).
- The highest mean **F1** was achieved by **{best_f1_row['model_name']}** ({best_f1_row['f1_mean']:.4f}).

These results should be interpreted as a multi-metric comparison rather than as evidence that one model dominates every criterion. In particular, ROC-AUC and PR-AUC measure ranking/discrimination across thresholds, whereas Recall and F1 depend on the fixed decision threshold used in the experiment.
"""

display(Markdown(rq1_analysis))


### RQ1 Analysis

- **XGBoost** achieved the best overall mean dataset rank (1.33).
- The highest mean **PR-AUC** was achieved by **XGBoost** (0.5725).
- The highest mean **ROC-AUC** was achieved by **XGBoost** (0.8841).
- The highest mean **Recall** was achieved by **LightGBM** (0.8189).
- The highest mean **F1** was achieved by **AdaBoost** (0.5389).

These results should be interpreted as a multi-metric comparison rather than as evidence that one model dominates every criterion. In particular, ROC-AUC and PR-AUC measure ranking/discrimination across thresholds, whereas Recall and F1 depend on the fixed decision threshold used in the experiment.


### 5.2 Final hold-out test performance

In [16]:
test_view = test_metrics[[
    "dataset_name", "model_name", "roc_auc", "pr_auc", "recall", "f1"
]].copy()
for col in ["roc_auc", "pr_auc", "recall", "f1"]:
    test_view[col] = test_view[col].round(4)

display(test_view.sort_values(["dataset_name", "model_name"]))

,dataset_name,model_name,roc_auc,pr_auc,recall,f1
6,Heart Disease Health Indicators (BRFSS 2015),AdaBoost,0.8326,0.3676,0.7687,0.3788
8,Heart Disease Health Indicators (BRFSS 2015),LightGBM,0.8387,0.3768,0.7938,0.3793
7,Heart Disease Health Indicators (BRFSS 2015),XGBoost,0.8385,0.3756,0.7959,0.3771
3,Heart Failure Prediction,AdaBoost,0.9256,0.9394,0.8529,0.8832
5,Heart Failure Prediction,LightGBM,0.9187,0.9191,0.8529,0.8700
4,Heart Failure Prediction,XGBoost,0.9223,0.9186,0.8235,0.8528
0,Personal Key Indicators of Heart Disease (2022),AdaBoost,0.8695,0.3860,0.6705,0.3657
2,Personal Key Indicators of Heart Disease (2022),LightGBM,0.8849,0.4180,0.7712,0.3276
1,Personal Key Indicators of Heart Disease (2022),XGBoost,0.8850,0.4181,0.7684,0.3248


The hold-out test set is not used during cross-validation. Similar CV and hold-out results provide an additional check that the reported comparison is not driven only by a favorable validation fold.

## 6. RQ2 - Stability across cross-validation folds

RQ2 examines how sensitive each model's predictive performance is to the specific training/validation partition. Stability is summarized using the standard deviation and range across the five folds. Smaller values indicate more consistent fold-to-fold performance.

In [17]:
stability_cols = [
    "dataset_name", "model_name",
    "roc_auc_mean", "roc_auc_std", "roc_auc_range",
    "pr_auc_mean", "pr_auc_std", "pr_auc_range",
    "recall_mean", "recall_std", "recall_range",
    "f1_mean", "f1_std", "f1_range",
]

stability_table = cv_summary[stability_cols].copy()
metric_cols = [c for c in stability_table.columns if c not in ["dataset_name", "model_name"]]
stability_table[metric_cols] = stability_table[metric_cols].round(4)

display(stability_table.sort_values(["dataset_name", "model_name"]))

,dataset_name,model_name,roc_auc_mean,roc_auc_std,roc_auc_range,pr_auc_mean,pr_auc_std,pr_auc_range,recall_mean,recall_std,recall_range,f1_mean,f1_std,f1_range
6,Heart Disease Health Indicators (BRFSS 2015),AdaBoost,0.8324,0.0031,0.0070,0.3678,0.0078,0.0192,0.7727,0.0082,0.0219,0.3826,0.0030,0.0065
7,Heart Disease Health Indicators (BRFSS 2015),LightGBM,0.8378,0.0023,0.0049,0.3797,0.0059,0.0148,0.7955,0.0047,0.0108,0.3822,0.0021,0.0045
8,Heart Disease Health Indicators (BRFSS 2015),XGBoost,0.8386,0.0027,0.0052,0.3797,0.0059,0.0138,0.8010,0.0046,0.0108,0.3812,0.0016,0.0042
3,Heart Failure Prediction,AdaBoost,0.9242,0.0419,0.1126,0.9227,0.0435,0.1179,0.8449,0.0294,0.0741,0.8578,0.0190,0.0437
4,Heart Failure Prediction,LightGBM,0.9259,0.0335,0.0801,0.9161,0.0376,0.0968,0.8843,0.0223,0.0617,0.8801,0.0212,0.0559
5,Heart Failure Prediction,XGBoost,0.9263,0.0333,0.0840,0.9244,0.0338,0.0830,0.8745,0.0368,0.0967,0.8755,0.0231,0.0570
0,Personal Key Indicators of Heart Disease (2022),AdaBoost,0.8715,0.0042,0.0104,0.3853,0.0087,0.0209,0.6794,0.0106,0.0285,0.3765,0.0082,0.0213
1,Personal Key Indicators of Heart Disease (2022),LightGBM,0.8871,0.0034,0.0075,0.4133,0.0097,0.0233,0.7770,0.0103,0.0237,0.3351,0.0029,0.0076
2,Personal Key Indicators of Heart Disease (2022),XGBoost,0.8873,0.0033,0.0072,0.4136,0.0083,0.0186,0.7798,0.0077,0.0159,0.3324,0.0025,0.0068


### 6.1 Fold-level ROC-AUC values

In [18]:
roc_fold_table = fold_metrics.pivot_table(
    index=["dataset_name", "model_name"],
    columns="fold",
    values="roc_auc"
).round(4)
roc_fold_table.columns = [f"Fold {int(c)}" for c in roc_fold_table.columns]

display(roc_fold_table)

Fold 1  Fold 2  \
dataset_name                                    model_name                   
Heart Disease Health Indicators (BRFSS 2015)    AdaBoost    0.8347  0.8357   
                                                LightGBM    0.8395  0.8391   
                                                XGBoost     0.8407  0.8405   
Heart Failure Prediction                        AdaBoost    0.9271  0.8547   
                                                LightGBM    0.9332  0.8668   
                                                XGBoost     0.9340  0.8681   
Personal Key Indicators of Heart Disease (2022) AdaBoost    0.8774  0.8684   
                                                LightGBM    0.8919  0.8847   
                                                XGBoost     0.8922  0.8850   

                                                            Fold 3  Fold 4  \
dataset_name                                    model_name                   
Heart Disease Health Indicators (BRFSS 2015)    AdaBoost    0.8296  0.8287   
                                                LightGBM    0.8357  0.8349   
                                                XGBoost     0.8359  0.8355   
Heart Failure Prediction                        AdaBoost    0.9411  0.9673   
                                                LightGBM    0.9383  0.9469   
                                                XGBoost     0.9420  0.9521   
Personal Key Indicators of Heart Disease (2022) AdaBoost    0.8738  0.8669   
                                                LightGBM    0.8894  0.8851   
                                                XGBoost     0.8892  0.8852   

                                                            Fold 5  
dataset_name                                    model_name          
Heart Disease Health Indicators (BRFSS 2015)    AdaBoost    0.8333  
                                                LightGBM    0.8398  
                                                XGBoost     0.8404  
Heart Failure Prediction                        AdaBoost    0.9310  
                                                LightGBM    0.9445  
                                                XGBoost     0.9354  
Personal Key Indicators of Heart Disease (2022) AdaBoost    0.8713  
                                                LightGBM    0.8844  
                                                XGBoost     0.8850

In [19]:
# Identify the most and least stable dataset/model combinations by ROC-AUC standard deviation.
stability_rank = cv_summary[["dataset_name", "model_name", "roc_auc_std", "pr_auc_std"]].copy()
most_stable = stability_rank.sort_values("roc_auc_std").iloc[0]
least_stable = stability_rank.sort_values("roc_auc_std", ascending=False).iloc[0]

# Dataset-level average variability helps compare the three datasets.
dataset_variability = (
    cv_summary.groupby("dataset_name", as_index=False)
    .agg(
        mean_roc_auc_std=("roc_auc_std", "mean"),
        mean_pr_auc_std=("pr_auc_std", "mean"),
    )
    .sort_values("mean_roc_auc_std")
)

display(dataset_variability.round(4))

rq2_analysis = f"""
### RQ2 Analysis

- The most stable model/dataset combination by ROC-AUC standard deviation is **{most_stable['model_name']}** on **{most_stable['dataset_name']}** (std = {most_stable['roc_auc_std']:.4f}).
- The largest ROC-AUC fold variation occurs for **{least_stable['model_name']}** on **{least_stable['dataset_name']}** (std = {least_stable['roc_auc_std']:.4f}).
- Stability should be interpreted together with dataset size and class distribution. A smaller dataset may be more sensitive to which observations are assigned to each fold, although sample size is not necessarily the only explanation for variation.

Overall, the fold-level statistics make it possible to distinguish models that achieve strong average performance from models whose results are highly sensitive to a particular cross-validation split.
"""

display(Markdown(rq2_analysis))

,dataset_name,mean_roc_auc_std,mean_pr_auc_std
0,Heart Disease Health Indicators (BRFSS 2015),0.0027,0.0065
2,Personal Key Indicators of Heart Disease (2022),0.0036,0.0089
1,Heart Failure Prediction,0.0362,0.0383



### RQ2 Analysis

- The most stable model/dataset combination by ROC-AUC standard deviation is **LightGBM** on **Heart Disease Health Indicators (BRFSS 2015)** (std = 0.0023).
- The largest ROC-AUC fold variation occurs for **AdaBoost** on **Heart Failure Prediction** (std = 0.0419).
- Stability should be interpreted together with dataset size and class distribution. A smaller dataset may be more sensitive to which observations are assigned to each fold, although sample size is not necessarily the only explanation for variation.

Overall, the fold-level statistics make it possible to distinguish models that achieve strong average performance from models whose results are highly sensitive to a particular cross-validation split.


## 7. Experiment summary

This notebook provides the reproducible evidence for the model-comparison component of the study:

- all three models use the same processed inputs and fixed experimental configuration;
- RQ1 is evaluated with ROC-AUC, PR-AUC, Recall, F1, within-dataset ranks, and cross-dataset averages;
- RQ2 is evaluated using fold-level results and variability statistics such as standard deviation and range;
- final hold-out test results are reported separately from cross-validation;
- reusable implementation remains in `src/`, while this notebook records experiment execution and interpretation.

The SHAP-based feature-ranking stability analysis for **RQ3** is outside the model-comparison scope of this notebook and can be integrated separately.